# WEEK-1
### NO NEED TO UPLOAD ANY DATASET

In [47]:
# !pip install -q tensorflow opencv-python numpy pandas matplotlib scikit-learn

import os
import random
import ntpath

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import cv2

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Dropout, Flatten
from tensorflow.keras.optimizers import Adam

if not os.path.exists('data'):
    !git clone https://github.com/Aman-py/data.git
else:
    print("Local 'data' directory already present.")

columns = ['center', 'left', 'right', 'steering', 'throttle', 'reverse', 'speed']

datadir = 'data'

dataset = pd.read_csv(
    os.path.join(datadir, 'driving_log.csv'),
    names=columns
)

dataset.head()

print("Number of samples:", len(dataset))

print("\nColumns:")
print(dataset.columns)

# Clean image paths
def removePath(path):
    base, tail = ntpath.split(path)
    return tail


Local 'data' directory already present.
Number of samples: 7259

Columns:
Index(['center', 'left', 'right', 'steering', 'throttle', 'reverse', 'speed'], dtype='object')


In [48]:
dataset.head()

,center,left,right,steering,throttle,reverse,speed
0,/home/aman-py/Desktop/Self-Driving-Car/data/IM...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,0.0,0.0,0.0,0.000078
1,/home/aman-py/Desktop/Self-Driving-Car/data/IM...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,0.0,0.0,0.0,0.000078
2,/home/aman-py/Desktop/Self-Driving-Car/data/IM...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,0.0,0.0,0.0,0.000079
3,/home/aman-py/Desktop/Self-Driving-Car/data/IM...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,0.0,0.0,0.0,0.000078
4,/home/aman-py/Desktop/Self-Driving-Car/data/IM...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,/home/aman-py/Desktop/Self-Driving-Car/data/I...,0.0,0.0,0.0,0.000079


#  WEEK-2

In [49]:
# WEEK - 2
# a. Dash-com, images and steering analysis dataset

dataset['center'] = dataset['center'].apply(removePath)
dataset['left'] = dataset['left'].apply(removePath)
dataset['right'] = dataset['right'].apply(removePath)

dataset.head()

,center,left,right,steering,throttle,reverse,speed
0,center_2018_10_17_16_36_30_865.jpg,left_2018_10_17_16_36_30_865.jpg,right_2018_10_17_16_36_30_865.jpg,0.0,0.0,0.0,0.000078
1,center_2018_10_17_16_36_30_966.jpg,left_2018_10_17_16_36_30_966.jpg,right_2018_10_17_16_36_30_966.jpg,0.0,0.0,0.0,0.000078
2,center_2018_10_17_16_36_31_098.jpg,left_2018_10_17_16_36_31_098.jpg,right_2018_10_17_16_36_31_098.jpg,0.0,0.0,0.0,0.000079
3,center_2018_10_17_16_36_31_253.jpg,left_2018_10_17_16_36_31_253.jpg,right_2018_10_17_16_36_31_253.jpg,0.0,0.0,0.0,0.000078
4,center_2018_10_17_16_36_31_366.jpg,left_2018_10_17_16_36_31_366.jpg,right_2018_10_17_16_36_31_366.jpg,0.0,0.0,0.0,0.000079


In [50]:
# Analyze steering angles

num_bins = 25

hist, bins = np.histogram(dataset['steering'], num_bins)

print("Histogram:")
print(hist)

print("\nBin edges:")
print(bins)

Histogram:
[  48   16   13   15   21   36   48   69  161  174  337  239 5454   56
  160   82   97   99   37   25   32   15   12    5    8]

Bin edges:
[-1.   -0.92 -0.84 -0.76 -0.68 -0.6  -0.52 -0.44 -0.36 -0.28 -0.2  -0.12
 -0.04  0.04  0.12  0.2   0.28  0.36  0.44  0.52  0.6   0.68  0.76  0.84
  0.92  1.  ]


In [51]:
# Calculate bin centers

center = (bins[:-1] + bins[1:]) * 0.5

print(center)

[-0.96 -0.88 -0.8  -0.72 -0.64 -0.56 -0.48 -0.4  -0.32 -0.24 -0.16 -0.08
  0.    0.08  0.16  0.24  0.32  0.4   0.48  0.56  0.64  0.72  0.8   0.88
  0.96]


In [ ]:
# Plot steering-angle distribution

threshold = 500

plt.figure(figsize=(10, 5))

plt.bar(center, hist, width=0.05)

plt.xticks(
    np.linspace(-1, 1, 25),
    rotation=90
)

plt.title("Steering Angles")
plt.xlabel("Steering Angle")
plt.ylabel("Number of Images")

x1 = np.min(dataset['steering'])
x2 = np.max(dataset['steering'])

plt.plot(
    (x1, x2),
    (threshold, threshold)
)

plt.show()

In [ ]:
# Remove excessive steering-angle samples

remove_list = []

for i in range(num_bins):

    bin_indices = []

    for j in range(len(dataset['steering'])):

        if (
            dataset['steering'].iloc[j] >= bins[i]
            and dataset['steering'].iloc[j] <= bins[i + 1]
        ):
            bin_indices.append(j)

    bin_indices = shuffle(
        bin_indices,
        random_state=42
    )

    # Remove samples above threshold
    if len(bin_indices) > threshold:

        bin_indices = bin_indices[threshold:]

        remove_list.extend(bin_indices)

print("Samples to remove:", len(remove_list))

In [ ]:
# Balance the dataset

dataset = dataset.drop(dataset.index[remove_list])

dataset = dataset.reset_index(drop=True)

print("Dataset size after balancing:", len(dataset))

In [ ]:
# Check the balanced steering distribution

hist, _ = np.histogram(
    dataset['steering'],
    num_bins
)

plt.figure(figsize=(10,5))

plt.bar(
    center,
    hist,
    width=0.05
)

plt.xticks(
    np.linspace(-1, 1, 25),
    rotation=90
)

plt.title("Steering Angles After Balancing")

plt.xlabel("Steering Angle")

plt.ylabel("Number of Images")

plt.axhline(
    threshold,
    linestyle='--'
)

plt.show()

In [ ]:
dataset.iloc[1]

In [ ]:
# Load image paths and steering values

def loadImageSteering(datadir, dataset):

    imagePath = []
    steeringPath = []

    for i in range(len(dataset)):

        center_image = dataset.iloc[i]['center']
        steering = float(dataset.iloc[i]['steering'])

        imagePath.append(
            os.path.join(datadir, center_image)
        )

        steeringPath.append(steering)

    imagePath = np.asarray(imagePath)
    steeringPath = np.asarray(steeringPath)

    return imagePath, steeringPath

In [ ]:
imagePath, steeringPath = loadImageSteering(
    os.path.join(datadir, 'IMG'),
    dataset
)

print("First image:")
print(imagePath[0])

print("\nNumber of images:", len(imagePath))

print("Number of steering values:", len(steeringPath))

print(os.path.exists(imagePath[0]))

In [ ]:
# Train-test split

x_train, x_test, y_train, y_test = train_test_split(
    imagePath,
    steeringPath,
    test_size=0.2,
    random_state=6
)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    y_train,
    bins=num_bins,
    width=0.05
)

plt.xticks(
    np.linspace(-1, 1, 25),
    rotation=45
)

plt.title("Training Dataset")
plt.xlabel("Steering Angle")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    y_test,
    bins=num_bins,
    width=0.05
)

plt.xticks(
    np.linspace(-1, 1, 25),
    rotation=45
)

plt.title("Testing Dataset")
plt.xlabel("Steering Angle")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
# Mean baseline model

mean_steering = np.mean(y_train)

baseline_predictions = np.full(
    len(y_test),
    mean_steering
)

baseline_mse = np.mean(
    (y_test - baseline_predictions) ** 2
)

print("Mean steering angle:", mean_steering)
print("Baseline MSE:", baseline_mse)

# WEEK-3

In [ ]:
# Deep learning model - Deep learning for regression

def imagePreprocessing(img_path):

    # Read image
    img = mpimg.imread(img_path)

    # Crop the image
    # Remove sky and car hood
    img = img[60:135, :, :]

    # Convert RGB to YUV
    img = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)

    # Apply Gaussian blur
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Resize to NVIDIA model input size
    img = cv2.resize(img, (200, 66))

    # Normalize pixel values
    img = img / 255.0

    return img

In [ ]:
sample_image = x_train[0]

original = mpimg.imread(sample_image)

processed = imagePreprocessing(sample_image)

fig, axs = plt.subplots(1, 2, figsize=(15, 6))

axs[0].imshow(original)
axs[0].set_title("Original Image")
axs[0].axis("off")

axs[1].imshow(processed)
axs[1].set_title("Preprocessed Image")
axs[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sample = imagePreprocessing(x_train[0])

print("Image shape:", sample.shape)
print("Steering angle:", y_train[0])

In [ ]:
def batch_generator(image_paths, steering_values, batch_size=32):

    while True:

        # Shuffle the dataset every epoch
        indices = np.random.permutation(len(image_paths))

        for start in range(0, len(image_paths), batch_size):

            # Select indices for this batch
            batch_indices = indices[start:start + batch_size]

            images = []
            steering_angles = []

            for index in batch_indices:

                # Load and preprocess image
                image = imagePreprocessing(image_paths[index])

                images.append(image)
                steering_angles.append(steering_values[index])

            # Convert to NumPy arrays
            X = np.array(images, dtype=np.float32)
            Y = np.array(steering_angles, dtype=np.float32)

            yield X, Y


batch_size = 32

train_generator = batch_generator(
    x_train,
    y_train,
    batch_size
)

validation_generator = batch_generator(
    x_test,
    y_test,
    batch_size
)

X_batch, Y_batch = next(train_generator)

print("Batch image shape:", X_batch.shape)
print("Batch steering shape:", Y_batch.shape)

In [ ]:
def nvidiaModel():

    model = Sequential()

    # Layer 1
    model.add(
        Conv2D(
            24,
            (5, 5),
            strides=(2, 2),
            input_shape=(66, 200, 3),
            activation='elu'
        )
    )

    # Layer 2
    model.add(
        Conv2D(
            36,
            (5, 5),
            strides=(2, 2),
            activation='elu'
        )
    )

    # Layer 3
    model.add(
        Conv2D(
            48,
            (5, 5),
            strides=(2, 2),
            activation='elu'
        )
    )

    # Layer 4
    model.add(
        Conv2D(
            64,
            (3, 3),
            activation='elu'
        )
    )

    # Layer 5
    model.add(
        Conv2D(
            64,
            (3, 3),
            activation='elu'
        )
    )

    # Dropout
    model.add(Dropout(0.5))

    # Convert feature maps into one-dimensional vector
    model.add(Flatten())

    # Fully connected layers
    model.add(Dense(100, activation='elu'))

    model.add(Dropout(0.5))

    model.add(Dense(50, activation='elu'))

    model.add(Dropout(0.5))

    model.add(Dense(10, activation='elu'))

    model.add(Dropout(0.5))

    # Final regression output
    model.add(Dense(1))

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='mse'
    )

    return model

In [ ]:
model = nvidiaModel()

model.summary()

#WEEK-4

In [ ]:
# Calculate training steps

steps_per_epoch = len(x_train) // batch_size
validation_steps = len(x_test) // batch_size

print("Steps per epoch:", steps_per_epoch)
print("Validation steps:", validation_steps)

In [ ]:
# Train the model

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    epochs=30,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Prepare test images

x_test_processed = np.array(
    [imagePreprocessing(img) for img in x_test],
    dtype=np.float32
)

print("Test data shape:", x_test_processed.shape)

In [ ]:
# Evaluate the model

test_loss = model.evaluate(
    x_test_processed,
    y_test,
    verbose=1
)

print("Test MSE:", test_loss)

In [ ]:
# Predict steering angles

predictions = model.predict(
    x_test_processed,
    verbose=1
)

predictions = predictions.flatten()

print("First 10 actual values:")
print(y_test[:10])

print("\nFirst 10 predicted values:")
print(predictions[:10])

In [ ]:
# Compare actual and predicted values

comparison = pd.DataFrame({
    'Actual Steering': y_test,
    'Predicted Steering': predictions
})

comparison.head(10)

In [ ]:
# Visualize actual vs predicted steering

plt.figure(figsize=(15, 6))

plt.plot(
    y_test[:100],
    label='Actual Steering'
)

plt.plot(
    predictions[:100],
    label='Predicted Steering'
)

plt.title("Actual vs Predicted Steering Angles")
plt.xlabel("Test Sample")
plt.ylabel("Steering Angle")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Visualize images with predictions

plt.figure(figsize=(10, 7))

for i in range(6):

    plt.subplot(2, 3, i + 1)

    image = mpimg.imread(x_test[i])

    plt.imshow(image)

    plt.axis('off')

    plt.title(
        f"Actual: {y_test[i]:.3f}\n"
        f"Predicted: {predictions[i]:.3f}"
    )

plt.tight_layout()
plt.show()

In [ ]:
# Calculate baseline vs CNN performance

print("Mean Baseline MSE:", baseline_mse)
print("NVIDIA CNN Test MSE:", test_loss)